# Study in Germany Assistant: run
Runtime → Change runtime type → **T4 GPU**. Start from an empty GPU
(Runtime → Disconnect and delete runtime), then run the cells from top to bottom, **once each**.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = "/content/drive/MyDrive/nlp-tasks-2026/Assignment 1"   # change only this line if the folder is elsewhere
%cd "$PROJECT/rag"

# docling is only needed to rebuild the database from the PDFs and is slow to install: skipped here
!grep -v docling ../requirements.txt > /tmp/requirements_run.txt
!pip install -q -r /tmp/requirements_run.txt

# The refusal check needs its 3 calibration files in data/ (copied from testing/ if missing; -n keeps existing ones)
!cp -n "../testing/Test Results/json/"refusal_*.json ../data/ 2>/dev/null; ls ../data/refusal_*.json

## 2. Model and Hugging Face login

In [ ]:
import os

# Generator Qwen3.5-4B, stored on Google Drive
os.environ["RAG_MODEL_PATH"] = "/content/drive/MyDrive/models/Qwen3.5-4B"

# Hugging Face login, needed for SPLADE (accept the licence on huggingface.co/naver/splade-v3 first)
from huggingface_hub import get_token, login
if get_token() is None:
    login()

## 3. Start the assistant (chat page + `POST /ask`)
Run once per session. A link to the chat page appears below the cell.

In [ ]:
from pipeline import RAGPipeline
from api import start_in_background

rag = RAGPipeline(use_splade=True,              # SPLADE as a third search (False: BM25 + dense only)
                  use_reranker=True,            # re-rank 20 candidates, keep the top k (False: off)
                  refusal_method="combined",    # 2-of-3 vote before answering ("none" | "bm25" | "dense" | "llm_judge")
                  retriever="both")             # BM25 + dense ("bm25" | "dense")
start_in_background(rag)

## 4. Ask from the notebook (optional)

In [ ]:
thinking, answer = rag.conduct_rag("What are the requirements for a blocked account in Germany?")
print(answer)